In [1]:
import redback
print(redback.__version__)

No module named 'lalsimulation'
lalsimulation is not installed. Some EOS based models will not work. Please use bilby eos or pass your own EOS generation class to the model
11:50 bilby INFO    : Running bilby version: 2.3.0
11:50 redback INFO    : Running redback version: 1.12.1


1.12.1


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import redback.interaction_processes as ip
import redback.sed as sed
import redback.photosphere as photosphere
from astropy.cosmology import Planck18 as cosmo 
import astropy.units as uu

import extinction
from extinction import ccm89, fitzpatrick99, apply, remove
from scipy.interpolate import RegularGridInterpolator
import sncosmo

In [3]:
def sn1998bw_template(time, redshift, amplitude, **kwargs):
    """
    A wrapper to the SN1998bw template. Only valid between 1100-11000 Angstrom and 0.01 to 90 days post explosion in rest frame

    Parameters
    ----------
    time
        time in days in observer frame (post explosion)
    redshift
        redshift
    amplitude
        amplitude scaling factor, where 1.0 is the original brightness of SN1998bw; and f_lambda is scaled by this factor
    kwargs
        Additional keyword arguments required by redback.
    frequency
        Required if output_format is 'flux_density'. frequency to calculate - Must be same length as time array or a single number).
    bands
        Required if output_format is 'magnitude' or 'flux'.
    output_format
        'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'
    cosmology
        Cosmology to use for luminosity distance calculation. Defaults to Planck18. Must be a astropy.cosmology object.
    Returns
    -------
        set by output format - 'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'

    """
    import sncosmo
    model = sncosmo.Model(source='v19-1998bw')
    original_redshift = 0.0085
    cosmology = kwargs.get("cosmology", cosmo)
    original_dl = (43*uu.Mpc).to(uu.cm).value

    # From roughly matching to Galama+ or Clocchiatti+1998bw light curves
    original_peak_time = 15
    model.set(z=original_redshift, t0=original_peak_time)
    model.set_source_peakmag(14.25, band='bessellb', magsys='ab')
    tts = np.geomspace(0.01, 90, 200)
    lls = np.linspace(1620, 11000, 300)
    f_lambda = model.flux(tts, lls) #erg/s/cm^2/Angstrom.
    l_lambda = f_lambda * 4 * np.pi * original_dl**2  # erg/s/Angstrom

    # We consider this the rest frame spectrum of 1998bw. Now we can redshift it and scale it.
    time_obs = tts * (1 + redshift)
    lambda_obs = lls * (1 + redshift)
    dl_new = cosmology.luminosity_distance(redshift).cgs.value
    f_lambda_obs = l_lambda / (4 * np.pi * dl_new**2)
    f_lambda_obs = amplitude * f_lambda_obs * (1 + redshift) # accounting for bandwidth stretching
    f_lambda_obs = f_lambda_obs * uu.erg / uu.s / uu.cm ** 2 / uu.Angstrom
    if kwargs['output_format'] == 'flux_density':
        frequency = kwargs['frequency']
        # work in obs frame
        ff_array = lambda_to_nu(lambda_obs)

        # Convert flux density to mJy
        fmjy = f_lambda_obs.to(uu.mJy, equivalencies=uu.spectral_density(wav=lambda_obs * uu.Angstrom)).value
        # Create interpolator on obs frame grid
        flux_interpolator = RegularGridInterpolator(
            (time_obs, ff_array),
            fmjy,
            bounds_error=False,
            fill_value=0.0)

        # Prepare points for interpolation
        if isinstance(frequency, (int, float)):
            frequency = np.ones_like(time) * frequency

        # Create points for evaluation
        points = np.column_stack((time, frequency))

        # Return interpolated flux density with (1+z) correction for observer frame
        return flux_interpolator(points)
    elif kwargs['output_format'] == 'spectra':
        return namedtuple('output', ['time', 'lambdas', 'spectra'])(time=time_obs,
                                                                    lambdas=lambda_obs,
                                                                    spectra=f_lambda_obs)
    else:
        return sed.get_correct_output_format_from_spectra(time=time, time_eval=time_obs,
                                                          spectra=f_lambda_obs, lambda_array=lambda_obs,
                                                          **kwargs)


In [27]:
#GRB 060318
event_name = '060218'
AB_mag = 17.22
redshift = 0.0331
epoch = 11.0  # Days in observer frame
band = 'bessellr'
a_v = 0.39 #true value from literature 
time = 11 #observed time in days 
frequency = 3.0e14#Hz central frequency in given band -- needs to be within certain wavelength range 

#doesnt work 
frequency_obs = 1.02780e+14 * (1 + redshift) #r-band Hz

print(f"{frequency_obs:.3e}")
#this helps but now my fraction is huge /11 

#only works at 1.0e15 ?? why ?? and 3.0e14 Hz - between thesevalues 
#WHY ??? this is the limitation of the model - in a narrow angstrom range - converted to frequency 

1.062e+14


In [23]:
def lambda_to_nu(wavelength_angstrom):
    """ Converts wavelength in Angstroms to frequency in Hz """
    c = 299792458  # speed of light in m/s
    return c / (wavelength_angstrom * 1e-10)


#doing this wrong ?? def lambda_to_nu(wavelength_angstrom):
    # Convert wavelength in Angstroms to frequency in Hz
    c = 2.99792458e18  # speed of light in Angstrom/s
    return c / wavelength_angstrom 

#nevermind just use redback one for now 

In [24]:
SN_1998bw = sn1998bw_template(
    time = time,
    redshift=redshift,
    amplitude=1.0,
    output_format = 'flux_density',
    frequency = frequency
)

f_1998bw_interp = SN_1998bw[0]

print("Flux density of 1998bw =", f_1998bw_interp)

#surely we expect flux density to be bigger than this ? 

Flux density of 1998bw = 0.4236890942070455


In [7]:
#STEP 2: Convert GRB event AB magnitude into flux density using redback 
def calc_flux_density_from_ABmag(AB_mag):
    """
    Calculate flux density from AB magnitude assuming monochromatic AB filter

    :param magnitudes: AB magnitude values
    :return: flux density
    """
    return (AB_mag * uu.ABmag).to(uu.mJy)

flux_density_event = calc_flux_density_from_ABmag(AB_mag)

print("This is the flux density of  GRB event", event_name ,f"without extinction correction: {flux_density_event:.3f}")

#check this using online calculator -- CORRECT -- OK TO PROCEED TO NEXT STEP 

#STEP 3: Use fitzpatrick99 extinction function (if applicable) to find de-reddened value for GRB flux density 
#to get wavelength for GRB event, use redback tables 
#wavelength must be an array 
wavelength =  np.array([6498.09000]) #angstroms -- r-band wavelength 
dereddened_flux_event = extinction.remove(fitzpatrick99(wavelength, a_v, 3.1),flux_density_event) 

#dereddened flux is now a 1-d array so display the first element 
print(f"This is the extinction corrected GRB event flux density found using fitzpatrick99: {dereddened_flux_event[0]:.3f}")


This is the flux density of  GRB event 060218 without extinction correction: 0.470 mJy
This is the extinction corrected GRB event flux density found using fitzpatrick99: 0.620 mJy


In [8]:
print(f"Flux density at day {time}: {f_1998bw_interp} mJy")

#SKIPPED STEP 6 -- INTERPOLATED IN ONE GO 

'''FINAL RATIO:'''
#STEP 7: take value from STEP 3 and STEP 6 in ratio with one another to get final result of flux ratio 

f_1998bw_ratio = dereddened_flux_event / f_1998bw_interp

print(f"The final flux density ratio result of F_GRB / F_1998bw = {f_1998bw_ratio[0]:.3f}")


Flux density at day 11: 0.05631593541776886 mJy
The final flux density ratio result of F_GRB / F_1998bw = 11.010 mJy


In [9]:
#GAVIN trialling model for different redshifts: 
''' at z = 0.01, 69mJy '''

' at z = 0.01, 69mJy '